# ETL Revenue Sharing 00570 - PER ORDER
> Kirim uang masuk (estimated income) **per order** outlet 00570 - DISTRICT 2 MEIKARTA ke API landlord (sumber: `order_est_income_for_landlord`).

Created at : 2026-11-20

Created by lahiardhan@jiwagroup.com

LATEST UPDATE: 2026-11-20 : tanggal dibuang dari TransactionNumber (mall memotong nomor > 50 char), cek panjang <= 50 sebelum POST, cek jumlah success vs TotalSuccess setelah POST

LATEST UPDATE: 2026-11-20 : TransactionNumber pakai order_reference + payment type (sama kayak Remarks), kunci guard = TransactionNumber tanpa HHMMSS

LATEST UPDATE: 2026-11-20 : TransactionNumber & Remarks tanpa prefix Janji_Jiwa_ (mulai dari jilid)

LATEST UPDATE: 2026-11-20 : guard anti-dobel per transaction_id (HHMMSS diabaikan saat cek log) + dump log per tanggal langsung setelah POST

**Catatan:**
- `Amount` = uang masuk penuh per baris (`subtotal - sales_type_commission`); 10% dihitung sendiri oleh landlord.
- `TransactionNumber = <jilid>_<order_reference>_<PaymentType>_<HHMMSS>` (tanpa prefix `Janji_Jiwa_`, tanpa tanggal; tanggal ada di field `TransactionDate` & `Remarks`). HHMMSS = jam run, dipakai supaya kiriman revisi punya nomor beda.
- **Mall memotong `TransactionNumber` jadi maksimal 50 karakter** (kejadian 2026-11-20, batch 1553125) -> notebook STOP kalau ada nomor > 50 char.
- Split payment dengan payment type SAMA di 1 order -> nomor kembar -> notebook STOP (cek sumber dulu).
- `Remarks = <jilid>_<order_reference>_<PaymentType>_<YYYY-MM-DD>_<HHMMSS>`.
- `DRY_RUN = True` (default): hanya hitung & tampilkan payload, TIDAK kirim ke API dan TIDAK nulis log.

**Guard anti-dobel** (cek ke `api_landlord_log_detail`, kunci = `TransactionNumber` tanpa `_HHMMSS`; baris log per order = yang diawali `<jilid>_`):

| Status di log | Aksi |
|---|---|
| `success` | skip |
| `error` + `duplicate` | skip (mall sudah punya) |
| `timeout_error` | skip + peringatan (belum pasti sampai ke mall). Kirim hanya kalau sudah dicek manual: `RESEND_TIMEOUT_ERROR = True` |
| `http_error`, `error` lain | kirim ulang (mall menolak, belum masuk) |
| tidak ada di log | kirim |

- Tanggal yang sudah diterima mall dalam format lama (per payment type) -> STOP (kalau dikirim lagi per order = dobel).
- Log disimpan **per tanggal, langsung setelah POST** (summary + detail satu transaksi DB). Kalau simpan log gagal -> STOP, tanggal berikutnya tidak dikirim.
- Revisi (kurang/lebih Rp) TIDAK lewat notebook ini (akan di-skip guard): kirim lewat notebook revisi dengan nomor dasar sama + HHMMSS baru + `Remarks` diawali `Revise-`.

# IMPORT LIBRARY

In [49]:
from datetime import datetime, timedelta
import mysql.connector
from mysql.connector import Error
from mysql.connector import errorcode
import time

from pytz import timezone

import pandas as pd
import numpy as np

import pymysql

import importlib, importlib.util

import warnings
warnings.filterwarnings("ignore")

from sqlalchemy import create_engine
import paramiko

import requests
import json

In [50]:
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)

# CONFIG DB

In [51]:
# Import Config Data
import importlib, importlib.util

def module_from_file(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

config_dw = module_from_file("config_dw", "../../db_config/config_db_datawarehouse.py")

# CONFIG

In [52]:
DRY_RUN = False                 # True = tidak kirim ke API & tidak nulis log; False = kirim beneran
RESEND_TIMEOUT_ERROR = False   # True HANYA setelah cek manual bahwa transaksi timeout belum masuk ke mall
JILID = '00570'
LOAD_DATA_BY = 'lahiardhan@jiwagroup.com'
API_URL = 'https://revenueapi.lippomalls.com:8443/api/revenue/v1/save'

# range tanggal yang mau dikirim (start_date = tanggal cutover, jangan sebelum itu)
start_date_str = '2026-01-02'
end_date_str = '2026-01-02'

print("DRY_RUN              :", DRY_RUN)
print("RESEND_TIMEOUT_ERROR :", RESEND_TIMEOUT_ERROR)
print("start_day            :", start_date_str)
print("end_day              :", end_date_str)

DRY_RUN              : False
RESEND_TIMEOUT_ERROR : False
start_day            : 2026-01-02
end_day              : 2026-01-02


# MAIN PROGRESS
### 1. Extract estimated income per order

In [53]:
def get_est_income_per_order(start_date_str, end_date_str):
    conn = mysql.connector.connect(host=config_dw.hostname,
                             port=config_dw.port,
                             user=config_dw.username,
                             password=config_dw.password,
                             db=config_dw.db_target)
    query = """
        SELECT date, outlet_code, order_id, transaction_id, order_reference, payment_type,
               subtotal, tax_amount, total_sales, sales_type_commission
        FROM order_est_income_for_landlord
        WHERE outlet_code = '{2}'
            AND date BETWEEN '{0}' AND '{1}'
        ORDER BY transaction_date
        """.format(start_date_str, end_date_str, JILID)

    df_est = pd.read_sql(query, conn)
    conn.close()
    return df_est

est_income = get_est_income_per_order(start_date_str, end_date_str)
print("rows:", len(est_income), "| orders:", est_income['order_id'].nunique())
est_income.head()

rows: 39 | orders: 39


,date,outlet_code,order_id,transaction_id,order_reference,payment_type,subtotal,tax_amount,total_sales,sales_type_commission
0,2026-01-02,00570,3861848b-28c4-4bef-bd2f-478a772e00d4,8fd17ddd-37de-4876-b452-d18273518d2b,GF/GF-580/PHPK.020126,Grab Food,28428.6077,2874.0000,31303.0000,4264.2912
1,2026-01-02,00570,be34ae63-acf6-4bd1-b72e-2516a42c79bc,72f92b27-526c-4396-8545-29c09298abbb,#2630-6177,iSeller QRIS,16363.6364,1636.3636,18000.0000,114.5455
2,2026-01-02,00570,267d0ea2-ed91-4e31-8621-c6fb5f812b74,87879726-5bd5-4d25-b056-6ea628271c51,GF/GF-196/9I9E.020126,Grab Food,50425.8804,5074.0000,55500.0000,7563.8821
3,2026-01-02,00570,0f8d22b9-d31e-4d30-8f13-1bec80c02db5,0a204e54-40ad-4094-bfd6-04e5afa608e1,TA/BRYAN/SQ2W.020126,Jiwa+,18181.8182,1818.0000,20000.0000,0.0000
4,2026-01-02,00570,652557bd-1127-4ff4-ba20-ec70b8967bc1,fd7eaa09-01c4-4e0f-a578-df662ea5eff5,#2630-6178,iSeller QRIS,20000.0000,1999.9999,22000.0000,140.0000


### 2. Transform (payload per order)

In [54]:
est = est_income.copy()
for col in ['subtotal', 'tax_amount', 'total_sales', 'sales_type_commission']:
    est[col] = pd.to_numeric(est[col]).astype(float)
est['date'] = est['date'].astype(str)

# order dengan total_sales 0 tidak dikirim (sama kayak versi per payment type: channel tanpa sales di-skip)
zero_rows = est[est['total_sales'] == 0]
print("baris total_sales = 0 (di-skip):", len(zero_rows))
est = est[est['total_sales'] != 0]

run_ts = pd.Timestamp.now().strftime("%H%M%S")

final_data = pd.DataFrame({
    'jilid': est['outlet_code'].astype(str),
    'TransactionDate': est['date'],
    'PaymentType': est['payment_type'],
    'TotalSales': est['total_sales'],
    'TotalTax': est['tax_amount'],
    # uang masuk penuh per baris; dibulatkan 4 desimal hanya di output yang dikirim
    'Amount': (est['subtotal'] - est['sales_type_commission']).round(4),
})
# TransactionNumber = order_reference + payment type (split payment beda payment type tetap beda nomor), HHMMSS = jam run (buat revisi)
final_data['TransactionNumber'] = (final_data['jilid'] + '_' + est['order_reference'].astype(str)
                                   + '_' + final_data['PaymentType'].astype(str) + '_' + run_ts)
final_data['_key'] = final_data['TransactionNumber'].str[:-7]   # kunci guard = nomor tanpa _HHMMSS
final_data['Remarks'] = ('Testing_' + final_data['jilid'] + '_' + est['order_reference'].astype(str)
                         + '_' + final_data['PaymentType'].astype(str) + '_' + final_data['TransactionDate'] + '_' + run_ts)

# fail loud: TransactionNumber harus unik & <= 50 char (mall memotong nomor yang lebih panjang -> respons gak cocok lagi dgn log kita)
MAX_TN_LEN = 50
dup = final_data[final_data['TransactionNumber'].duplicated(keep=False)]
if not dup.empty:
    print(dup[['TransactionDate', 'PaymentType', 'TransactionNumber']])
    raise ValueError(f"{len(dup)} TransactionNumber dobel (1 order dibayar 2x dgn payment type sama?), cek sumber dulu.")
max_len = final_data['TransactionNumber'].str.len().max()
if max_len > MAX_TN_LEN:
    print(final_data.loc[final_data['TransactionNumber'].str.len() > MAX_TN_LEN, ['TransactionDate', 'PaymentType', 'TransactionNumber']])
    raise ValueError(f"TransactionNumber terpanjang {max_len} char > {MAX_TN_LEN} (mall memotongnya). Pendekkan format dulu.")

print("rows to send:", len(final_data))
print(final_data.groupby('TransactionDate')['Amount'].agg(['count', 'sum']))
final_data.head()

baris total_sales = 0 (di-skip): 0
rows to send: 39
                 count          sum
TransactionDate                    
2026-01-02          39 1757501.3051


,jilid,TransactionDate,PaymentType,TotalSales,TotalTax,Amount,TransactionNumber,_key,Remarks
0,00570,2026-01-02,Grab Food,31303.0000,2874.0000,24164.3165,00570_GF/GF-580/PHPK.020126_Grab Food_164235,00570_GF/GF-580/PHPK.020126_Grab Food,Testing_00570_GF/GF-580/PHPK.020126_Grab Food_...
1,00570,2026-01-02,iSeller QRIS,18000.0000,1636.3636,16249.0909,00570_#2630-6177_iSeller QRIS_164235,00570_#2630-6177_iSeller QRIS,Testing_00570_#2630-6177_iSeller QRIS_2026-01-...
2,00570,2026-01-02,Grab Food,55500.0000,5074.0000,42861.9983,00570_GF/GF-196/9I9E.020126_Grab Food_164235,00570_GF/GF-196/9I9E.020126_Grab Food,Testing_00570_GF/GF-196/9I9E.020126_Grab Food_...
3,00570,2026-01-02,Jiwa+,20000.0000,1818.0000,18181.8182,00570_TA/BRYAN/SQ2W.020126_Jiwa+_164235,00570_TA/BRYAN/SQ2W.020126_Jiwa+,Testing_00570_TA/BRYAN/SQ2W.020126_Jiwa+_2026-...
4,00570,2026-01-02,iSeller QRIS,22000.0000,1999.9999,19860.0000,00570_#2630-6178_iSeller QRIS_164235,00570_#2630-6178_iSeller QRIS,Testing_00570_#2630-6178_iSeller QRIS_2026-01-...


In [55]:
final_data

,jilid,TransactionDate,PaymentType,TotalSales,TotalTax,Amount,TransactionNumber,_key,Remarks
0,00570,2026-01-02,Grab Food,31303.0000,2874.0000,24164.3165,00570_GF/GF-580/PHPK.020126_Grab Food_164235,00570_GF/GF-580/PHPK.020126_Grab Food,Testing_00570_GF/GF-580/PHPK.020126_Grab Food_...
1,00570,2026-01-02,iSeller QRIS,18000.0000,1636.3636,16249.0909,00570_#2630-6177_iSeller QRIS_164235,00570_#2630-6177_iSeller QRIS,Testing_00570_#2630-6177_iSeller QRIS_2026-01-...
2,00570,2026-01-02,Grab Food,55500.0000,5074.0000,42861.9983,00570_GF/GF-196/9I9E.020126_Grab Food_164235,00570_GF/GF-196/9I9E.020126_Grab Food,Testing_00570_GF/GF-196/9I9E.020126_Grab Food_...
3,00570,2026-01-02,Jiwa+,20000.0000,1818.0000,18181.8182,00570_TA/BRYAN/SQ2W.020126_Jiwa+_164235,00570_TA/BRYAN/SQ2W.020126_Jiwa+,Testing_00570_TA/BRYAN/SQ2W.020126_Jiwa+_2026-...
4,00570,2026-01-02,iSeller QRIS,22000.0000,1999.9999,19860.0000,00570_#2630-6178_iSeller QRIS_164235,00570_#2630-6178_iSeller QRIS,Testing_00570_#2630-6178_iSeller QRIS_2026-01-...
5,00570,2026-01-02,iSeller QRIS,50000.0000,4545.4545,45136.3637,00570_#2630-6179_iSeller QRIS_164235,00570_#2630-6179_iSeller QRIS,Testing_00570_#2630-6179_iSeller QRIS_2026-01-...
6,00570,2026-01-02,Jiwa+,23834.0000,2166.0000,21667.2727,00570_TA/ANISSA/9WVU.020126_Jiwa+_164235,00570_TA/ANISSA/9WVU.020126_Jiwa+,Testing_00570_TA/ANISSA/9WVU.020126_Jiwa+_2026...
7,00570,2026-01-02,Jiwa+,107000.0000,9784.0000,97215.3972,00570_DEL/WIDYA/MUC6.020126_Jiwa+_164235,00570_DEL/WIDYA/MUC6.020126_Jiwa+,Testing_00570_DEL/WIDYA/MUC6.020126_Jiwa+_2026...
8,00570,2026-01-02,Jiwa+,19800.0000,1800.0000,18000.0000,00570_TA/UNAA/X861.020126_Jiwa+_164235,00570_TA/UNAA/X861.020126_Jiwa+,Testing_00570_TA/UNAA/X861.020126_Jiwa+_2026-0...
9,00570,2026-01-02,Grab Food,37500.0000,3438.0000,28952.9075,00570_GF/GF-213/6XXG.020126_Grab Food_164235,00570_GF/GF-213/6XXG.020126_Grab Food,Testing_00570_GF/GF-213/6XXG.020126_Grab Food_...


### 3. Guard anti-dobel

In [56]:
# ambil SEMUA log jilid ini (tabel kecil): per order dicek per nomor (tanpa HHMMSS), format lama dicek per tanggal
conn = mysql.connector.connect(host=config_dw.hostname, port=config_dw.port, user=config_dw.username,
                               password=config_dw.password, db=config_dw.db_resource)
log_all = pd.read_sql("""
    SELECT transaction_number, transaction_date, payment_type, amount, remarks, status, error_message, created_at
    FROM api_landlord_log_detail
    WHERE jilid = '{0}'
    """.format(JILID), conn)
conn.close()
print("log rows jilid", JILID, ":", len(log_all))

log rows jilid 00570 : 402


In [57]:
# kunci = transaction_number tanpa _HHMMSS di belakang (revisi/run beda jam tetap kena)
# baris log per order = diawali '<jilid>_' dan diakhiri _HHMMSS; selain itu (Janji_Jiwa_..., 01065-...) = format lama per payment type
is_per_order = log_all['transaction_number'].str.startswith(JILID + '_') & log_all['transaction_number'].str.contains(r'_\d{6}$')
# buang _HHMMSS, lalu _YYYYMMDD kalau ada (format test 2026-11-20 masih pakai tanggal) -> kunci = <jilid>_<order_reference>_<PaymentType>
log_all['key'] = (log_all['transaction_number'].str.replace(r'_\d{6}$', '', regex=True)
                  .str.replace(r'_\d{8}$', '', regex=True).where(is_per_order))
log_all['transaction_date'] = log_all['transaction_date'].astype(str)
is_success = log_all['status'] == 'success'
is_duplicate = log_all['error_message'].fillna('').str.strip().eq('duplicate')   # sama kayak DAG: TRIM(error_message) = 'duplicate'
is_timeout = log_all['status'] == 'timeout_error'
log_all['accepted'] = is_success | is_duplicate   # pasti sudah ada di mall

# 1) tanggal yang sudah diterima mall dalam format lama (per payment type) -> STOP, kalau dikirim lagi per order = dobel
legacy = log_all[log_all['key'].isna() & log_all['accepted'] & log_all['transaction_date'].isin(set(final_data['TransactionDate']))]
if legacy.empty:
    print("OK: tidak ada kiriman format lama (per payment type) di tanggal-tanggal ini.")
else:
    print("PERINGATAN: tanggal ini sudah diterima mall dalam format lama (per payment type):")
    print(legacy.groupby('transaction_date')['amount'].agg(['count', 'sum']))
    if not DRY_RUN:
        raise ValueError("Range berisi tanggal yang sudah dikirim per payment type. Naikkan start_date ke tanggal cutover.")

# 2) sudah pasti di mall (success / duplicate) -> skip
accepted_keys = set(log_all.loc[log_all['accepted'] & log_all['key'].notna(), 'key'])
already = final_data['_key'].isin(accepted_keys)
print("sudah di mall (success/duplicate) -> skip:", int(already.sum()))

# 3) pernah timeout & belum pernah pasti diterima -> belum tentu sampai ke mall -> skip + peringatan (kecuali RESEND_TIMEOUT_ERROR)
timeout_keys = set(log_all.loc[is_timeout & log_all['key'].notna(), 'key']) - accepted_keys
uncertain = final_data['_key'].isin(timeout_keys)
if uncertain.any():
    print(f"PERINGATAN: {int(uncertain.sum())} transaksi pernah timeout_error, belum pasti sudah masuk ke mall:")
    print(final_data.loc[uncertain, ['TransactionDate', 'PaymentType', 'Amount', 'TransactionNumber']])
    if RESEND_TIMEOUT_ERROR:
        print("RESEND_TIMEOUT_ERROR = True -> tetap dikirim (pastikan sudah dicek manual).")
    else:
        print("-> skip. Cek manual ke landlord, lalu set RESEND_TIMEOUT_ERROR = True kalau memang belum masuk.")
skip = already | (uncertain & (not RESEND_TIMEOUT_ERROR))

# 4) 'duplicate' tanpa pernah 'success' di log kita -> info (mall menganggap sudah ada, cek kalau muncul di kiriman pertama)
success_keys = set(log_all.loc[is_success & log_all['key'].notna(), 'key'])
dup_only = log_all[is_duplicate & log_all['key'].notna() & ~log_all['key'].isin(success_keys)]
if not dup_only.empty:
    print(f"INFO: {dup_only['key'].nunique()} transaksi ditolak 'duplicate' tanpa pernah success di log kita:")
    print(dup_only[['transaction_date', 'transaction_number', 'payment_type', 'amount', 'created_at']])

# sisanya (belum ada di log / http_error / error lain) dikirim
final_data = final_data[~skip].drop(columns='_key').reset_index(drop=True)
print("rows to send after guard:", len(final_data))
if not final_data.empty:
    print(final_data.groupby('TransactionDate')['Amount'].agg(['count', 'sum']))

OK: tidak ada kiriman format lama (per payment type) di tanggal-tanggal ini.
sudah di mall (success/duplicate) -> skip: 5
rows to send after guard: 34
                 count          sum
TransactionDate                    
2026-01-02          34 1636184.0812


In [41]:
final_data

,jilid,TransactionDate,PaymentType,TotalSales,TotalTax,Amount,TransactionNumber,Remarks
0,00570,2026-01-02,Grab Food,31303.0000,2874.0000,24164.3165,00570_GF/GF-580/PHPK.020126_Grab Food_162604,Testing_00570_GF/GF-580/PHPK.020126_Grab Food_...
1,00570,2026-01-02,iSeller QRIS,18000.0000,1636.3636,16249.0909,00570_#2630-6177_iSeller QRIS_162604,Testing_00570_#2630-6177_iSeller QRIS_2026-01-...
2,00570,2026-01-02,Grab Food,55500.0000,5074.0000,42861.9983,00570_GF/GF-196/9I9E.020126_Grab Food_162604,Testing_00570_GF/GF-196/9I9E.020126_Grab Food_...
3,00570,2026-01-02,Jiwa+,20000.0000,1818.0000,18181.8182,00570_TA/BRYAN/SQ2W.020126_Jiwa+_162604,Testing_00570_TA/BRYAN/SQ2W.020126_Jiwa+_2026-...
4,00570,2026-01-02,iSeller QRIS,22000.0000,1999.9999,19860.0000,00570_#2630-6178_iSeller QRIS_162604,Testing_00570_#2630-6178_iSeller QRIS_2026-01-...


### 4. Kirim ke API + simpan log PER TANGGAL

Urutan per tanggal: POST -> simpan log (summary + detail, 1 transaksi DB) -> baru lanjut tanggal berikutnya. Simpan log gagal = STOP.

In [42]:
def access_token(url, username, password):
    body = {'grant_type': 'password', 'username': username, 'password': password}
    response = requests.post(url, body)
    if response.status_code == 200:
        return response.json().get("access_token")
    raise ValueError(f"Gagal ambil token: {response.status_code} {response.text[:200]}")   # fail loud


MAX_ATTEMPTS = 2
TIMEOUT_SECONDS = 90
BACKOFF_SECONDS = 5
DETAIL_COLS = ['jilid', 'TransactionNumber', 'TransactionDate', 'PaymentType', 'TotalSales', 'TotalTax', 'Amount', 'Remarks']
DETAIL_NAMES = ['jilid', 'transaction_number', 'transaction_date', 'payment_type', 'total_sales', 'total_tax', 'amount', 'remarks']


def post_one_date(group, token):
    """POST semua baris 1 tanggal (1 batch). Return (log_summary, log_detail); logika sama kayak notebook 00570 lama."""
    revenue_data = group[['TransactionNumber', 'TransactionDate', 'Amount', 'Remarks']].to_dict('records')
    request_payload = {"revenueDatas": revenue_data}
    headers = {"Content-Type": "application/json", "Authorization": f"Bearer {token}"}

    response, request_error = None, None
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            response = requests.post(API_URL, json=request_payload, headers=headers, timeout=TIMEOUT_SECONDS)
            break
        except requests.exceptions.RequestException as e:
            request_error = e
            print(f"RETRY | {group['TransactionDate'].iloc[0]} | attempt {attempt} failed: {str(e)}")
            if attempt < MAX_ATTEMPTS:
                time.sleep(BACKOFF_SECONDS)

    now = datetime.now()
    detail = group[DETAIL_COLS].copy()
    detail.columns = DETAIL_NAMES
    summary = {'revenue_batch_id': None, 'jilid': JILID, 'transaction_date': group['TransactionDate'].iloc[0],
               'total_records_sent': len(revenue_data), 'total_success': 0, 'total_error': len(revenue_data),
               'http_status_code': None, 'api_code': None, 'api_message': None, 'api_url': API_URL,
               'request_payload': json.dumps(request_payload), 'response_payload': None, 'created_at': now}

    if response is None:
        error_msg = f"Timeout/ConnectionError after {MAX_ATTEMPTS} attempts: {str(request_error)}"
        summary.update({'api_message': error_msg, 'response_payload': error_msg})
        detail['mall_transaction_id'] = None
        detail['revenue_batch_id'] = None
        detail['status'] = 'timeout_error'
        detail['error_message'] = error_msg
    elif response.status_code == 200:
        try:
            resp_json = response.json()
        except ValueError:
            resp_json = {'Message': f'Non-JSON 200 body: {response.text[:500]}', 'transactionReturnDatas': [], 'ErrorTransactionNumber': []}
        summary.update({'revenue_batch_id': resp_json.get('RevenueBatchId'), 'total_success': resp_json.get('TotalSuccess'),
                        'total_error': resp_json.get('TotalError'), 'http_status_code': response.status_code,
                        'api_code': resp_json.get('Code'), 'api_message': resp_json.get('Message'),
                        'response_payload': json.dumps(resp_json)})
        response_details = pd.DataFrame(resp_json.get('transactionReturnDatas', []))
        if not response_details.empty:
            response_details = response_details.rename(columns={'TenantTransactionNumber': 'transaction_number',
                                                                'MallTransactionId': 'mall_transaction_id'})
            detail = detail.merge(response_details[['transaction_number', 'mall_transaction_id']], on='transaction_number', how='left')
        else:
            detail['mall_transaction_id'] = None
        error_map = {}
        for err in resp_json.get('ErrorTransactionNumber', []):
            txn_num, _, error_msg = str(err).partition('|')
            error_map[txn_num] = error_msg if error_msg else 'unknown error'
        detail['revenue_batch_id'] = resp_json.get('RevenueBatchId')
        detail['status'] = detail['mall_transaction_id'].apply(lambda x: 'success' if pd.notna(x) else 'error')
        detail['error_message'] = detail['transaction_number'].map(error_map)
    else:
        summary.update({'http_status_code': response.status_code, 'api_message': response.text[:500],
                        'response_payload': response.text[:500]})
        detail['mall_transaction_id'] = None
        detail['revenue_batch_id'] = None
        detail['status'] = 'http_error'
        detail['error_message'] = f"HTTP {response.status_code}: {response.text[:200]}"

    detail['created_at'] = now
    return pd.DataFrame([summary]), detail


if DRY_RUN:
    print("DRY_RUN = True -> tidak ada request ke API & tidak ada log. Contoh payload per baris:")
    print(final_data[['TransactionNumber', 'TransactionDate', 'Amount', 'Remarks']].head(5).to_dict('records'))
elif final_data.empty:
    print("Tidak ada data baru untuk dikirim.")
else:
    df_token = pd.read_csv('../get_token.csv')
    df_token['jilid'] = df_token['jilid'].astype(str).str.zfill(5)
    df_token = df_token[df_token['jilid'] == JILID].reset_index(drop=True)
    token = access_token(df_token['url'].iloc[0], df_token['email'].iloc[0], df_token['password'].iloc[0])

    engine = create_engine("mysql+pymysql://{user}:{pw}@{hostsname}/{db}".format(
        hostsname=config_dw.hostname, user=config_dw.username, pw=config_dw.password, db=config_dw.db_resource))

    dates = sorted(final_data['TransactionDate'].unique())
    for i, d in enumerate(dates, start=1):
        group = final_data[final_data['TransactionDate'] == d].reset_index(drop=True)
        log_summary, log_detail = post_one_date(group, token)
        log_summary['load_data_by'] = LOAD_DATA_BY
        log_detail['load_data_by'] = LOAD_DATA_BY

        # simpan log tanggal ini SEKARANG (sebelum tanggal berikutnya), summary + detail 1 transaksi
        try:
            with engine.begin() as conn:
                log_summary.to_sql('api_landlord_logs', con=conn, if_exists='append', index=False)
                log_detail.to_sql('api_landlord_log_detail', con=conn, if_exists='append', index=False)
        except Exception as e:
            # hasil POST tetap ditampilkan supaya bisa dicatat manual
            print(log_detail[['transaction_number', 'amount', 'status', 'mall_transaction_id', 'error_message']].to_string(index=False))
            raise RuntimeError(f"POST tanggal {d} SUDAH jalan tapi simpan log GAGAL: {e}. STOP, tanggal berikutnya tidak dikirim. "
                               "Catat hasil di atas ke api_landlord_log_detail sebelum kirim ulang.") from e

        counts = log_detail['status'].value_counts().to_dict()
        print(f"[{i}/{len(dates)}] {d} | sent {len(group)} | amount {group['Amount'].sum():,.4f} | {counts} | log tersimpan")

        # cek: jumlah success yang kecocokan di log kita HARUS sama dgn TotalSuccess dari mall
        # (kalau beda, log kita salah catat -> guard bisa kirim ulang yang sebenarnya sudah masuk -> STOP)
        api_total_success = log_summary['total_success'].iloc[0]
        if log_summary['http_status_code'].iloc[0] == 200 and int(api_total_success or 0) != int((log_detail['status'] == 'success').sum()):
            raise RuntimeError(f"Tanggal {d}: mall bilang TotalSuccess = {api_total_success}, tapi yang cocok di log kita "
                               f"{int((log_detail['status'] == 'success').sum())}. STOP. Cek response_payload di api_landlord_logs "
                               "dan betulkan status di api_landlord_log_detail sebelum kirim ulang.")

[1/1] 2026-01-02 | sent 5 | amount 121,317.2239 | {'success': 5} | log tersimpan
